In [1]:
# Table 1 
# Performance metrics determined for environmental images using the fractional weighted approach



In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import CLASSIFICATION_DATA, get_predictions_scores, get_18score_vector, classes54, classes18


In [2]:
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict


In [3]:
p_model_outputs = CLASSIFICATION_DATA / "4.MODEL_PREDICTIONS"

p_global_weights =CLASSIFICATION_DATA /"globalweights_f1scores_18cl.csv"


In [ ]:
class_sum_other = ['Other', 'Acacia', 'Acer', 'Alnus', 'Amarantaceae', 'Artemisia', 'Betulaceae', 'Brassicaceae', 'Carduus', 'Caryophyllaceae', 'Cichorioideae', 'Cyperaceae', 
               'Echium', 'Ericaceae', 'Fagus', 'Gallium', 'Juglans', 'Lamiaceae', 'Liliaceae', 'Moraceae', 'Morphotype1', 
               'Morphotype2', 'Myrtaceae', 'Platanus', 'PopulusSp', 'Ranunculaceae', 'Rhamnus', 'Rosaceae', 'Rumex', 'Salix', 'Sanguisorba', 'Tilia', 'Ulmus', 'Urtica',
                 'ViburnumSambucusTp', 'XanthiumAmbrosia']


In [6]:
# export weights per model for weights

df_w = pd.read_csv(p_global_weights )
print(len(list(df_w["classe"])))

weights_dict = {
    model: df_group.set_index("classe")["f1score"].to_dict()
    for model, df_group in df_w.groupby("cval")
}
sum_f1_by_model = df_w.groupby("cval")["f1score"].sum().to_dict()
print(weights_dict)


df_w.head()

180
{'fold1': {'Buxus': 0.988235294117647, 'Cupressaceae': 0.956989247311828, 'Fraxinus': 0.87719298245614, 'IndetBlurry': 0.953488372093023, 'IndetCovered': 0.893617021276596, 'Lycopodium': 0.990033222591362, 'NonPollen': 0.937062937062937, 'Olea': 0.939393939393939, 'Other': 0.947890818858561, 'Phillyrea': 0.862745098039216, 'Pinaceae': 0.994818652849741, 'Pistacia': 0.909090909090909, 'Plantago': 0.985507246376812, 'Poaceae': 0.989473684210526, 'QuercusDeciduous': 0.782608695652174, 'QuercusIlex': 0.943661971830986, 'VitisF': 0.982456140350877, 'VitisS': 0.914285714285714}, 'fold10': {'Buxus': 0.976190476190476, 'Cupressaceae': 0.966666666666667, 'Fraxinus': 0.84, 'IndetBlurry': 0.98876404494382, 'IndetCovered': 0.888888888888889, 'Lycopodium': 1.0, 'NonPollen': 0.957746478873239, 'Olea': 0.984126984126984, 'Other': 0.944444444444444, 'Phillyrea': 0.872727272727273, 'Pinaceae': 1.0, 'Pistacia': 0.978723404255319, 'Plantago': 0.957746478873239, 'Poaceae': 0.989583333333333, 'QuercusD

,cval,classe,f1score,counter
0,fold1,Buxus,0.988235,84
1,fold1,Cupressaceae,0.956989,91
2,fold1,Fraxinus,0.877193,28
3,fold1,IndetBlurry,0.953488,44
4,fold1,IndetCovered,0.893617,49


In [7]:
def get_df_cval_(classes18, count_true, dict_cm, cval_id):

    """  
    count_true = dictionnaire avec le key = classes, and values is the total true = nombre de ground truth
    dict_cm = dictionnaire avec le key = classes, and values = vecteur de 18 scores (fractional weighted) des 18 classes ordonnées comme dans liste classes18
    but c'est d'avoir pour chaque cval pour avoir mean se pour les tables de valeurs finales
    """

    li_df=[]
    for i, cl_i in enumerate(classes18):
        nb_groundtruth = count_true[cl_i]

        vec_cl_i = dict_cm[cl_i]
        if round(sum(vec_cl_i),2)!=round(nb_groundtruth,2):
            print(f"Warning: Sum of predicted scores for class {cl_i} does not match number of ground truth. Sum: {sum(vec_cl_i)}, Ground truth: {nb_groundtruth}")

        TP = vec_cl_i[i]       
        tot_pred = sum(vec[i] for vec in dict_cm.values())
        FN = nb_groundtruth - TP
        FP = tot_pred - TP

        rec = TP / nb_groundtruth if nb_groundtruth > 0 else np.nan
        prec = TP / tot_pred if tot_pred > 0 else np.nan
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else np.nan

        recall = rec * 100
        precision = prec * 100
        f1score = f1 * 100
        li_df+=[[cval_id, cl_i, recall, precision, f1score,  nb_groundtruth, TP, FN, FP, tot_pred]]

    df=pd.DataFrame(li_df, columns= ["cval_id", "class", "recall", "precision", "f1score", "nb_groundtruth", 'TP', "FN", "FP", "nb_pred"])
    #df.head()
    return df

In [9]:
# ici regarde sans faire la démarche du "weighted" fractionall 18 classes env

all_prclass_percval = []


for idx in [1,2,3,4,5,6,7,8,9,10]:
    cval_id = f'fold{idx}'

    d_allpred18= defaultdict(lambda: np.zeros(len(classes18)))
    count_true = {cl_i : 0 for cl_i in classes18}

    true, predscores, filenames = get_predictions_scores(cval_id, p_model_outputs, classes54, refenv="env")
 
    for e, filename_i in enumerate(filenames):
        true_i = true[e]
        vector_k_18, class_true = get_18score_vector(cval_id, true_i, predscores[e], weights_dict, sum_f1_by_model, classes18, classes54, class_sum_other)
        d_allpred18[class_true]+= vector_k_18
        count_true[class_true]+=1
    df_cvali=get_df_cval_(classes18, count_true, d_allpred18, cval_id)
    all_prclass_percval.append(df_cvali)

df_overall = pd.concat(all_prclass_percval, ignore_index=True)
print(sum(df_overall["nb_groundtruth"]))


df_overall

12540


,cval_id,class,recall,precision,f1score,nb_groundtruth,TP,FN,FP,nb_pred
0,fold1,QuercusDeciduous,69.515766,80.343396,74.538422,12,8.341892,3.658108,2.040905,10.382797
1,fold1,QuercusIlex,96.061608,88.800881,92.288657,68,65.321894,2.678106,8.238068,73.559961
2,fold1,Buxus,99.832138,97.726281,98.767986,84,83.858996,0.141004,1.951080,85.810076
3,fold1,Phillyrea,82.895124,85.522147,84.188147,26,21.552732,4.447268,3.648614,25.201346
4,fold1,Fraxinus,86.896425,84.172887,85.512976,28,24.330999,3.669001,4.574982,28.905981
...,...,...,...,...,...,...,...,...,...,...
175,fold10,Other,93.125230,94.372961,93.744944,201,187.181713,13.818287,11.160812,198.342525
176,fold10,IndetCovered,87.824675,87.214268,87.518407,49,43.034091,5.965909,6.308857,49.342948
177,fold10,IndetBlurry,98.864387,97.619722,98.238112,44,43.500330,0.499670,1.060676,44.561006
178,fold10,NonPollen,94.654048,95.151547,94.902146,142,134.408749,7.591251,6.848806,141.257554


In [11]:
# determine mean and 2SE over 10 cross-validation folds

n_folds = df_overall['cval_id'].nunique() # 10

agg_df = df_overall.groupby('class')[['precision', 'recall', 'f1score', 'nb_groundtruth']].agg(['mean', 'std'])

agg_df[('precision', 'twose_standerr')] = agg_df[('precision', 'std')] / np.sqrt(n_folds)*2
agg_df[('recall', 'twose_standerr')] = agg_df[('recall', 'std')] / np.sqrt(n_folds)*2
agg_df[('f1score', 'twose_standerr')] = agg_df[('f1score', 'std')] / np.sqrt(n_folds)*2
agg_df[('nb_groundtruth', 'twose_standerr')] = agg_df[('nb_groundtruth', 'std')] / np.sqrt(n_folds)*2
agg_df = agg_df.drop(columns=[('precision', 'std'), ('recall', 'std'), ('f1score', 'std'), ('nb_groundtruth', 'std')])

agg_df.columns = ['-'.join(map(str, col)).strip() for col in agg_df.columns.values]
agg_df=agg_df.reset_index()

agg_df

,class,precision-mean,recall-mean,f1score-mean,nb_groundtruth-mean,precision-twose_standerr,recall-twose_standerr,f1score-twose_standerr,nb_groundtruth-twose_standerr
0,Buxus,98.311908,99.393425,98.846718,83.7,0.535767,0.476363,0.356316,0.305505
1,Cupressaceae,94.801831,96.665584,95.703887,91.4,1.232880,1.101915,0.710316,0.326599
2,Fraxinus,82.021163,80.992558,81.411749,27.9,2.808299,2.408340,1.885975,0.466667
3,IndetBlurry,97.967201,96.926702,97.436391,44.0,1.079744,0.960905,0.842009,0.000000
4,IndetCovered,89.464785,82.248615,85.645636,48.7,1.215468,2.616287,1.496897,0.305505
5,Lycopodium,99.504132,99.389231,99.444470,149.9,0.378025,0.439589,0.270160,0.200000
6,NonPollen,94.054067,93.166976,93.597799,142.0,1.100301,1.089905,0.873987,0.000000
7,Olea,95.140186,94.999367,95.020360,32.6,2.680985,2.600356,2.233239,0.326599
8,Other,93.992754,93.146338,93.564113,203.7,0.726486,0.738670,0.624446,1.492202
9,Phillyrea,79.213348,82.653555,80.791972,26.3,2.753718,3.137718,2.206055,0.305505


In [13]:
# calcul deviation predictions VS ground truth (over full dataset, so all cval together)

df_inter = df_overall.copy()
df_sum = df_inter.groupby("class").agg({"nb_groundtruth": "sum", "nb_pred": "sum"}).reset_index()
df_sum["deviation"] = (df_sum["nb_pred"] - df_sum["nb_groundtruth"]) / df_sum["nb_groundtruth"] # * 100
df_sum

agg_df2 = pd.merge(agg_df, df_sum[["class", "deviation"]], on="class", how="left")


agg_df2

,class,precision-mean,recall-mean,f1score-mean,nb_groundtruth-mean,precision-twose_standerr,recall-twose_standerr,f1score-twose_standerr,nb_groundtruth-twose_standerr,deviation
0,Buxus,98.311908,99.393425,98.846718,83.7,0.535767,0.476363,0.356316,0.305505,0.011092
1,Cupressaceae,94.801831,96.665584,95.703887,91.4,1.232880,1.101915,0.710316,0.326599,0.020090
2,Fraxinus,82.021163,80.992558,81.411749,27.9,2.808299,2.408340,1.885975,0.466667,-0.009978
3,IndetBlurry,97.967201,96.926702,97.436391,44.0,1.079744,0.960905,0.842009,0.000000,-0.010435
4,IndetCovered,89.464785,82.248615,85.645636,48.7,1.215468,2.616287,1.496897,0.305505,-0.080296
5,Lycopodium,99.504132,99.389231,99.444470,149.9,0.378025,0.439589,0.270160,0.200000,-0.001117
6,NonPollen,94.054067,93.166976,93.597799,142.0,1.100301,1.089905,0.873987,0.000000,-0.009209
7,Olea,95.140186,94.999367,95.020360,32.6,2.680985,2.600356,2.233239,0.326599,-0.000618
8,Other,93.992754,93.146338,93.564113,203.7,0.726486,0.738670,0.624446,1.492202,-0.008917
9,Phillyrea,79.213348,82.653555,80.791972,26.3,2.753718,3.137718,2.206055,0.305505,0.046269


In [ ]:
# saving file (used later on to generate Confidence Intervals for predictions)

#agg_df2.to_csv(CLASSIFICATION_DATA / "perf_fracW18env_mean2se.csv", index=False)


In [14]:
# Format class names and Order as for the manuscript 


name_mapping = {
    'Acacia': r'$\it{Acacia}$', 'Acer': r'$\it{Acer}$', 'Alnus': r'$\it{Alnus}$',
    'Amarantaceae': 'Amarantaceae', 'Artemisia': r'$\it{Artemisia}$', 'Betulaceae': r'$\it{Betulaceae}$',
    'Brassicaceae': 'Brassicaceae', 'Buxus': r'$\it{Buxus}$',
    'Carduus': r'$\it{Carduus}$',  'Caryophyllaceae': 'Caryophyllaceae',
    'Cichorioideae': 'Cichorioideae',
   'Cupressaceae': "Cupressaceae", 'Cyperaceae': 'Cyperaceae',
    'Echium': r'$\it{Echium}$', 'Ericaceae': 'Ericaceae', 'Fagus': r'$\it{Fagus}$', 'Fraxinus' : r'$\it{Fraxinus}$', 'FraxinusExcelsior': r'$\it{Fraxinus~excelsior}$',
    'FraxinusOrnus': r'$\it{Fraxinus~ornus}$', 'Gallium': r'$\it{Galium}$', 
    'Helianthemum': r'$\it{Helianthemum}$', 'Hypericum': r'$\it{Hypericum}$', 'IlexAquifolium': r'$\it{Ilex~aquifolium}$',
    'IndetBlurry': 'Indeterminate (blurry)', 'IndetCovered': 'Indeterminate (covered)',
    'Juglans': r'$\it{Juglans}$', 'Lamiaceae': 'Lamiaceae','Liliaceae': 'Liliaceae',  'Lycopodium': r'$\it{Lycopodium}$',
    'Moraceae': 'Moraceae', 'Morphotype1' : 'Morphotype 1', 'Morphotype2' : "Morphotype 2", 
    'Myrtaceae': 'Myrtaceae', 'NonPollen': 'Non-pollen', 'Olea': r'$\it{Olea}$',
    'Other': 'Other taxa', 'Phillyrea': r'$\it{Phillyrea}$',
    'Pinaceae': 'Pinaceae', 'Pistacia': r'$\it{Pistacia}$', 'Plantago': r'$\it{Plantago}$',
    'Platanus': r'$\it{Platanus}$', 'Poaceae': 'Poaceae', 'PopulusSp': r'$\it{Populus}$',
    'QuercusDeciduous': r'$\it{Quercus~deciduous}$', 'QuercusIlex': r'$\it{Quercus~ilex}$',
    'Ranunculaceae': 'Ranunculaceae', 'Rhamnus': r'$\it{Rhamnus}$', 'Rosaceae': 'Rosaceae',
    'Rumex': r'$\it{Rumex}$', 'Salix': r'$\it{Salix}$', 'Sanguisorba': r'$\it{Sanguisorba}$',
    'Tilia': r'$\it{Tilia}$', 'Ulmus': r'$\it{Ulmus}$', 'Urtica': r'$\it{Urtica}$',
    "ViburnumSambucusTp" : r'$\it{Viburnum / Sambucus}$',
    'VitisF': r'$\it{Vitis}$ fertile', 'VitisS': r'$\it{Vitis}$ sterile', 'XanthiumAmbrosia':  r'$\it{Xanthium / Ambrosia}$',
}



agg_df2["classes2"] = agg_df2["class"]
agg_df2["class"] =[name_mapping[el] for el in list(agg_df2["class"])]
agg_df2['classes2'] = pd.Categorical(agg_df2['classes2'], categories=classes18, ordered=True)
agg_df2 = agg_df2.sort_values('classes2')

agg_df2

,class,precision-mean,recall-mean,f1score-mean,nb_groundtruth-mean,precision-twose_standerr,recall-twose_standerr,f1score-twose_standerr,nb_groundtruth-twose_standerr,deviation,classes2
14,$\it{Quercus~deciduous}$,74.780360,69.539085,71.714000,12.1,4.571751,5.234757,3.826565,0.200000,-0.063575,QuercusDeciduous
15,$\it{Quercus~ilex}$,91.029775,94.322832,92.605588,67.7,1.757390,0.986158,0.621515,0.305505,0.037507,QuercusIlex
0,$\it{Buxus}$,98.311908,99.393425,98.846718,83.7,0.535767,0.476363,0.356316,0.305505,0.011092,Buxus
9,$\it{Phillyrea}$,79.213348,82.653555,80.791972,26.3,2.753718,3.137718,2.206055,0.305505,0.046269,Phillyrea
2,$\it{Fraxinus}$,82.021163,80.992558,81.411749,27.9,2.808299,2.408340,1.885975,0.466667,-0.009978,Fraxinus
7,$\it{Olea}$,95.140186,94.999367,95.020360,32.6,2.680985,2.600356,2.233239,0.326599,-0.000618,Olea
1,Cupressaceae,94.801831,96.665584,95.703887,91.4,1.232880,1.101915,0.710316,0.326599,0.020090,Cupressaceae
11,$\it{Pistacia}$,92.058952,93.576667,92.713379,23.0,2.390325,2.738327,1.636045,0.000000,0.018427,Pistacia
13,Poaceae,97.431498,97.848367,97.630627,95.6,0.628735,0.937943,0.506821,0.326599,0.004389,Poaceae
12,$\it{Plantago}$,97.268458,97.760316,97.465156,34.5,1.598727,1.779471,0.917444,0.333333,0.005987,Plantago


In [28]:
latex_rows = []
print(f"class  & recall (%) & precision (%) & f1 (%) & deviation (%) & support \\", "\n")
for _, row in agg_df2.iterrows():
    class_name = row["class"]

    recall = f"{row['recall-mean']:.1f} ± {row['recall-twose_standerr']:.1f}"
    precision    = f"{row['precision-mean']:.1f} ± {row['precision-twose_standerr']:.1f}"
    f1        = f"{row['f1score-mean']:.1f} ± {row['f1score-twose_standerr']:.1f}"
    deviation = f"{float(row['deviation'])*100:.1f}"
    support   = f"{row['nb_groundtruth-mean']:.1f} ± {row['nb_groundtruth-twose_standerr']:.1f}"

    line = f"{class_name} & {recall} & {precision} & {f1} & {deviation} & {support} \\\\"
    latex_rows.append(line)

latex_table = "\n".join(latex_rows)
print(latex_table)

class  & recall (%) & precision (%) & f1 (%) & deviation (%) & support \ 

$\it{Quercus~deciduous}$ & 69.5 ± 5.2 & 74.8 ± 4.6 & 71.7 ± 3.8 & -6.4 & 12.1 ± 0.2 \\
$\it{Quercus~ilex}$ & 94.3 ± 1.0 & 91.0 ± 1.8 & 92.6 ± 0.6 & 3.8 & 67.7 ± 0.3 \\
$\it{Buxus}$ & 99.4 ± 0.5 & 98.3 ± 0.5 & 98.8 ± 0.4 & 1.1 & 83.7 ± 0.3 \\
$\it{Phillyrea}$ & 82.7 ± 3.1 & 79.2 ± 2.8 & 80.8 ± 2.2 & 4.6 & 26.3 ± 0.3 \\
$\it{Fraxinus}$ & 81.0 ± 2.4 & 82.0 ± 2.8 & 81.4 ± 1.9 & -1.0 & 27.9 ± 0.5 \\
$\it{Olea}$ & 95.0 ± 2.6 & 95.1 ± 2.7 & 95.0 ± 2.2 & -0.1 & 32.6 ± 0.3 \\
Cupressaceae & 96.7 ± 1.1 & 94.8 ± 1.2 & 95.7 ± 0.7 & 2.0 & 91.4 ± 0.3 \\
$\it{Pistacia}$ & 93.6 ± 2.7 & 92.1 ± 2.4 & 92.7 ± 1.6 & 1.8 & 23.0 ± 0.0 \\
Poaceae & 97.8 ± 0.9 & 97.4 ± 0.6 & 97.6 ± 0.5 & 0.4 & 95.6 ± 0.3 \\
$\it{Plantago}$ & 97.8 ± 1.8 & 97.3 ± 1.6 & 97.5 ± 0.9 & 0.6 & 34.5 ± 0.3 \\
$\it{Vitis}$ fertile & 97.5 ± 0.9 & 97.4 ± 1.0 & 97.5 ± 0.5 & 0.2 & 57.2 ± 0.3 \\
$\it{Vitis}$ sterile & 94.1 ± 2.9 & 93.6 ± 2.3 & 93.8 ± 1.8 & 0.6 & 18.3 ±